# GT-Free OCR Metrics — Dataset Exploration

This notebook demonstrates the two companion datasets released with the paper
**"GT-Free OCR Metrics: Reference-Free Evaluation via Render-and-Compare"**
(NeurIPS 2025 Datasets & Benchmarks Track).

| Dataset | HuggingFace repo |
|---|---|
| Render-and-Compare pairs | `gt-free-ocr-metrics/omnidocbench-render-compare` |
| Qwen OCR Log-Probabilities | `gt-free-ocr-metrics/omnidocbench-qwen-ocr-logprobs` |

**Runtime:** all cells complete in under 2 minutes (downloads ~20 MB of sample data).
**Colab:** click *Runtime → Run all* after opening.


In [ ]:
%pip install -q datasets huggingface_hub pillow matplotlib pandas


In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset
from huggingface_hub import hf_hub_download

RENDER_REPO   = "gt-free-ocr-metrics/omnidocbench-render-compare"
LOGPROBS_REPO = "gt-free-ocr-metrics/omnidocbench-qwen-ocr-logprobs"
SAMPLE_N = 4   # pages to download for visual inspection

def fetch_image(repo_id, path_in_repo):
    local = hf_hub_download(repo_id=repo_id, filename=path_in_repo, repo_type="dataset")
    return Image.open(local).convert("RGB")

def fetch_json(repo_id, path_in_repo):
    try:
        local = hf_hub_download(repo_id=repo_id, filename=path_in_repo, repo_type="dataset")
        with open(local) as f:
            return json.load(f)
    except Exception:
        return None


---
## 1. Render-and-Compare Dataset

`omnidocbench-render-compare` contains pre-rendered `masked_original.png` /
`reconstructed.png` image pairs for all five OCR extraction variants,
plus the raw OCR HTML and element JSON for each page.


In [ ]:
VARIANTS = ["ocr_all", "ocr_all_no_mask", "ocr_text", "ocr_formula", "ocr_table"]

rows = []
for v in VARIANTS:
    ds = load_dataset(RENDER_REPO, v, split="train")
    rows.append({"variant": v, "pages": len(ds), "columns": len(ds.features)})

pd.DataFrame(rows)


### 1.1 Masked original vs reconstructed — side-by-side

The pipeline masks non-target regions on the original scan and renders the OCR
output back to an image. A good OCR result produces a reconstruction that closely
matches the masked original.


In [ ]:
ds_all = load_dataset(RENDER_REPO, "ocr_all", split="train")

# Sample pages from different document categories
prefixes = ["PPT", "book", "academic", "research"]
samples, seen = [], set()
for row in ds_all:
    pid = row["page_id"]
    for p in prefixes:
        if p not in seen and pid.lower().startswith(p.lower()):
            samples.append(row); seen.add(p); break
    if len(samples) >= SAMPLE_N:
        break
if len(samples) < SAMPLE_N:
    samples = list(ds_all.select(range(SAMPLE_N)))

fig, axes = plt.subplots(len(samples), 2, figsize=(14, 4.5 * len(samples)))
for i, row in enumerate(samples):
    orig  = fetch_image(RENDER_REPO, row["masked_original"])
    recon = fetch_image(RENDER_REPO, row["reconstructed"])
    axes[i, 0].imshow(orig);  axes[i, 0].set_title(f"masked_original\n{row['page_id']}", fontsize=8)
    axes[i, 1].imshow(recon); axes[i, 1].set_title("reconstructed", fontsize=8)
    for ax in axes[i]: ax.axis("off")
plt.suptitle("ocr_all variant — masked original (left) vs reconstructed (right)", y=1.002)
plt.tight_layout(); plt.show()


### 1.2 Same page across variants

In [ ]:
first_pid = ds_all[0]["page_id"]
compare_variants = ["ocr_all", "ocr_all_no_mask", "ocr_text"]

fig, axes = plt.subplots(1, len(compare_variants), figsize=(5 * len(compare_variants), 6))
for ax, variant in zip(axes, compare_variants):
    ds_v = load_dataset(RENDER_REPO, variant, split="train")
    row = next((r for r in ds_v if r["page_id"] == first_pid), None)
    if row:
        ax.imshow(fetch_image(RENDER_REPO, row["reconstructed"]))
    ax.set_title(f"reconstructed\nvariant: {variant}", fontsize=9); ax.axis("off")
plt.suptitle(f"Page: {first_pid}", y=1.01)
plt.tight_layout(); plt.show()


### 1.3 OCR output — HTML and element JSON

In [ ]:
row = ds_all[0]
html_path = hf_hub_download(RENDER_REPO, row["ocr_html"], repo_type="dataset")
with open(html_path) as f:
    snippet = f.read()[:800]
print("=== OCR HTML (first 800 chars) ===")
print(snippet)
print("...")


In [ ]:
elements = fetch_json(RENDER_REPO, row["ocr_elements"])
if elements:
    print(f"Text elements on this page: {len(elements)}")
    print("\nFirst 3 elements:")
    for elem in elements[:3]:
        print(json.dumps(elem, ensure_ascii=False, indent=2))
else:
    print("No element JSON for this page.")


---
## 2. OCR Log-Probabilities Dataset

`omnidocbench-qwen-ocr-logprobs` provides per-page token-level and
bounding-box-level log-probabilities from Qwen3.5-122B-A10B on all 1 355 pages.
These serve as a reference-free confidence signal for text-quality metrics.


In [ ]:
ds_lp = load_dataset(LOGPROBS_REPO, split="train")
df_lp = ds_lp.to_pandas()
print(f"Rows: {len(df_lp)}  |  Columns: {list(df_lp.columns)}")
df_lp[["n_total_tokens","n_bboxes","logprob_mean","logprob_min",
       "logprob_max","shannon_entropy_mean"]].describe().round(4)


### 2.1 Distribution of log-probability and entropy

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df_lp["logprob_mean"], bins=60, color="#4f86c6", edgecolor="white", linewidth=0.4)
axes[0].set_xlabel("logprob_mean"); axes[0].set_ylabel("pages")
axes[0].set_title("Mean log-prob per page\n(closer to 0 = more confident)")

axes[1].hist(df_lp["shannon_entropy_mean"], bins=60, color="#e07b39", edgecolor="white", linewidth=0.4)
axes[1].set_xlabel("shannon_entropy_mean")
axes[1].set_title("Mean Shannon entropy per page\n(lower = more confident)")

axes[2].scatter(df_lp["logprob_mean"], df_lp["shannon_entropy_mean"],
                alpha=0.35, s=8, color="#555")
axes[2].set_xlabel("logprob_mean"); axes[2].set_ylabel("shannon_entropy_mean")
axes[2].set_title("Confidence vs uncertainty")

plt.tight_layout(); plt.show()


### 2.2 Element type coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_lp["n_total_tokens"], bins=60, color="#6abf69", edgecolor="white", linewidth=0.4)
axes[0].set_xlabel("n_total_tokens"); axes[0].set_ylabel("pages")
axes[0].set_title("OCR output length (tokens per page)")

bbox_means = df_lp[["n_text_bboxes","n_formula_bboxes","n_table_bboxes"]].mean()
axes[1].bar(["text","formula","table"], bbox_means.values,
            color=["#4f86c6","#e07b39","#6abf69"])
axes[1].set_ylabel("mean bboxes per page")
axes[1].set_title("Average detected bboxes by element type")

plt.tight_layout(); plt.show()

print(f"Pages with formula bboxes : {(df_lp['n_formula_bboxes'] > 0).sum()}")
print(f"Pages with table bboxes   : {(df_lp['n_table_bboxes']  > 0).sum()}")
print(f"Text-only pages           : {((df_lp['n_formula_bboxes']==0)&(df_lp['n_table_bboxes']==0)).sum()}")


---
## 3. DocSim Training Triplets

The `docsim_triplets` config contains 20 280 triplets used to fine-tune the
DocSim LoRA similarity head. Each triplet has an **anchor** (masked original),
a **positive** (reconstruction of the same page), and a **negative**
(reconstruction of a different page). Hungarian edit distances are provided
as supervision labels.


In [ ]:
ds_tri = load_dataset(RENDER_REPO, "docsim_triplets", split="train")
df_tri = ds_tri.to_pandas()
print(f"Total triplets: {len(df_tri)}")
df_tri[["anchor_ed","positive_ed","negative_ed"]].describe().round(4)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df_tri["positive_ed"], bins=60, alpha=0.7, label="positive (same page)", color="#4f86c6")
ax.hist(df_tri["negative_ed"], bins=60, alpha=0.7, label="negative (different page)", color="#e07b39")
ax.set_xlabel("Hungarian edit distance  (lower = better OCR fidelity)")
ax.set_ylabel("triplets")
ax.set_title("Edit-distance distribution: positive vs negative pairs")
ax.legend(); plt.tight_layout(); plt.show()


### 3.1 Sample triplets — anchor / positive / negative

In [ ]:
triplet_indices = [0, len(df_tri) // 3, 2 * len(df_tri) // 3]
keys    = ["anchor_path", "positive_path", "negative_path"]
labels  = ["anchor\n(masked original)", "positive\n(same-page recon)", "negative\n(diff-page recon)"]
ed_keys = ["anchor_ed", "positive_ed", "negative_ed"]

fig, axes = plt.subplots(len(triplet_indices), 3, figsize=(15, 5 * len(triplet_indices)))
for row_i, idx in enumerate(triplet_indices):
    row = df_tri.iloc[idx]
    for col_i, (key, label, edk) in enumerate(zip(keys, labels, ed_keys)):
        img = fetch_image(RENDER_REPO, row[key])
        axes[row_i, col_i].imshow(img)
        axes[row_i, col_i].set_title(f"{label}\ned={row[edk]:.3f}", fontsize=9)
        axes[row_i, col_i].axis("off")
plt.suptitle("DocSim triplets", y=1.01)
plt.tight_layout(); plt.show()


---
## 4. Cross-Dataset Exploration

Merging render-and-compare metadata with log-probabilities on `page_id` gives
a unified view: we can examine how the OCR model's internal confidence relates
to document category and output length.


In [ ]:
df_meta = load_dataset(RENDER_REPO, "ocr_all", split="train").to_pandas()
df_joined = df_meta[["page_id"]].merge(df_lp, on="page_id", how="inner")
print(f"Joined rows: {len(df_joined)}")
df_joined[["page_id","n_total_tokens","logprob_mean","shannon_entropy_mean"]].head(6)


In [ ]:
def infer_category(pid):
    pid_l = pid.lower()
    for cat, kw in [("Slides/PPT","ppt"),("Book","book"),("Academic paper","academic"),
                    ("Research report","research"),("Financial","financial"),
                    ("Newspaper","news"),("Exam","exam"),("Magazine","magazine")]:
        if kw in pid_l:
            return cat
    return "Other"

df_joined["category"] = df_joined["page_id"].apply(infer_category)

fig, ax = plt.subplots(figsize=(11, 5))
for cat, grp in df_joined.groupby("category"):
    ax.scatter(grp["logprob_mean"], grp["n_total_tokens"],
               label=cat, alpha=0.5, s=12)
ax.set_xlabel("logprob_mean  (Qwen confidence; closer to 0 = higher confidence)")
ax.set_ylabel("n_total_tokens  (OCR output length)")
ax.set_title("Per-page OCR confidence vs output length, coloured by document category")
ax.legend(markerscale=2, fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()
